### EDA: Credit Card Fraud Detection

**Бизнес-цель:** отсечь мошеннические транзакции; ложный «нормальный» прогноз по фроду хуже, чем лишняя ручная проверка.

**Метрики:** для класса Fraud — **Recall** (не пропускать фрод) и **F1** (баланс с шумом при дисбалансе).

**Что видно в данных:** сильный **дисбаланс** классов → в пайплайне **stratified train/test split** и **`class_weight=balanced`** при обучении.

**Почему модели:** **LogisticRegression** — простой интерпретируемый baseline на табличных признаках; **RandomForest** — нелинейности и взаимодействия без ручного feature engineering; обе подходят для числового табличного датасета.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "fraud_detection").exists())
sys.path.insert(0, str(ROOT / "src"))

from fraud_detection.config import DATA_RAW, TARGET_COLUMN
from fraud_detection.data_loader import load_raw

path = DATA_RAW / "creditcard.csv"
if not path.exists():
    raise FileNotFoundError(f"Put creditcard.csv in data/raw/. Expected {path}")
df = load_raw(path)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df[TARGET_COLUMN].value_counts(normalize=True)

In [ ]:
df.describe()

In [ ]:
sns.countplot(data=df, x=TARGET_COLUMN)
plt.title("Class imbalance (0=Normal, 1=Fraud)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x=TARGET_COLUMN, y="Amount")
plt.title("Amount по классу (0=Normal, 1=Fraud)")
plt.tight_layout()
plt.show()

In [ ]:
corr = df.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
from loguru import logger
fraud = df[df[TARGET_COLUMN] == 1]
normal = df[df[TARGET_COLUMN] == 0]
logger.info("Fraud: {}, Normal: {}, Imbalance ratio: {:.0f}:1", len(fraud), len(normal), len(normal)/max(len(fraud),1))